## Load the json files that are created in plot2_overleaf.ipynb 
### The file names are like : sg_pair_1-12.json ,...,

In [ ]:
f = glob.glob('*pair*.json')
with open(f[-1], 'r') as file:
    df = json.load(file)


['sg_pair_1-12.json',
 'sg_pair_1-2.json',
 'sg_pair_1-5.json',
 'sg_pair_1-8.json',
 'sg_pair_12-15.json',
 'sg_pair_12-63.json',
 'sg_pair_139-221.json',
 'sg_pair_139-225.json',
 'sg_pair_2-12.json',
 'sg_pair_221-225.json',
 'sg_pair_62-221.json',
 'sg_pair_63-139.json',
 'sg_pair_63-221.json',
 'sg_pair_63-225.json',
 'sg_pair_71-225.json']

In [ ]:
import os
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import read
from ase.visualize.plot import plot_atoms

# -----------------------
# Config
# -----------------------
#from step1 you know what pairs you got as most frequent, alternatively, look at the image saved from step before

CIF_FOLDER = r"cifs" # cif files are stored here
group1 = '225'
group2 = '71'
# -----------------------
# Load pair JSON
# -----------------------

# -----------------------
# Match formula pairs
# -----------------------
matched_pairs = []



# Precompute reduced formulas for group2
formula_group2 = {}
for cif in df[group2]:
    # cif already contains "cifs/mp-xxx.cif", so just join with parent folder
    path = os.path.join(CIF_FOLDER, cif.replace('/', os.sep))
    try:
        struct = Structure.from_file(path)
        formula_group2[path] = struct.composition.reduced_formula
    except:
        continue

# Match with group1 entries
for cif1 in df[group1]:
    path1 = os.path.join(CIF_FOLDER, cif1.replace('/', os.sep))
    try:
        struct1 = Structure.from_file(path1)
        formula1 = struct1.composition.reduced_formula
    except:
        continue

    for path2, formula2 in formula_group2.items():
        if formula1 == formula2:
            matched_pairs.append((path1, path2))

# -----------------------
# Helper functions
# -----------------------

def to_subscript(formula):
    sub_map = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    result = ""
    for c in formula:
        result += c.translate(sub_map) if c.isdigit() else c
    return result

def get_reduced_formula(cif_path):
    try:
        struct = CifParser(cif_path).get_structures()[0]
        return struct.composition.reduced_formula
    except:
        return "Unknown"

def load_structure(filename):
    try:
        return read(filename)
    except:
        try:
            struct = CifParser(filename).get_structures()[0]
            return AseAtomsAdaptor.get_atoms(struct)
        except:
            print("skipped:", filename.split("/")[-1])
            return None

def plot_cif_pair(cif1_path, cif2_path, outname):
    atoms1 = load_structure(cif1_path)
    atoms2 = load_structure(cif2_path)

    if atoms1 is None or atoms2 is None:
        return

    formula1 = to_subscript(get_reduced_formula(cif1_path))
    formula2 = to_subscript(get_reduced_formula(cif2_path))

    os.makedirs("overleaf_sg_pair", exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    plot_atoms(atoms1, ax=axes[0], rotation=('20x,10y'), show_unit_cell=2)
    plot_atoms(atoms2, ax=axes[1], rotation=('20x,10y'), show_unit_cell=2)

    mpid1 = os.path.basename(cif1_path).split(".")[0]
    mpid2 = os.path.basename(cif2_path).split(".")[0]

    axes[0].set_title(f"{mpid1}\n{formula1}", fontsize=22)
    axes[1].set_title(f"{mpid2}\n{formula2}", fontsize=22)
    axes[0].tick_params(labelsize=22)
    axes[1].tick_params(labelsize=22)
    plt.tight_layout()
    plt.savefig(f"overleaf_sg_pair/{outname}.png", dpi=300)
    plt.close()

# -----------------------
# Generate image outputs
# -----------------------
# Only save specific matched pairs
allowed_names = {'mp-1183151_mp-1095936.png', 'mp-1183165_mp-1093918.png', 'mp-1183187_mp-1096207.png'}

for cif1, cif2 in matched_pairs:
    name1 = os.path.splitext(os.path.basename(cif1))[0]
    name2 = os.path.splitext(os.path.basename(cif2))[0]
    outname = f"{name1}_{name2}"
    
    # Check if this pair should be saved
    if f"{outname}.png"  in allowed_names:
        plot_cif_pair(cif1, cif2, outname)


C:\Users\moons\AppData\Local\Temp\ipykernel_5172\1965431697.py:72: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  struct = CifParser(cif_path).get_structures()[0]
